In [1]:
import pandas as pd
import numpy as np
from pathlib import Path

In [2]:
RAW = Path("data/dataset.csv")
OUT = Path("data/CleanedData.csv")

In [3]:
# Load the dataset and remove the unwanted CSV index column
data = pd.read_csv(RAW)
data = data.drop(columns=["Unnamed: 0"], errors="ignore")
data = data.reset_index(drop=True)
data["_row_id"] = data.index  # unique tie-breaker for deterministic sorting

In [4]:
# Normalize text values and replace placeholder strings with NaN
text_cols = data.select_dtypes(include=["object", "string"]).columns.tolist()
fake_nulls = ["", " ", "none", "null", "nan", "n/a", "na", "unknown", "?", "-"]

for col in text_cols:
    data[col] = data[col].str.strip().str.lower().replace(fake_nulls, np.nan)

# Drop rows only when one or more required text fields are missing
required_text = ["track_id", "artists", "album_name", "track_name", "track_genre"]
data = data.dropna(subset=required_text).reset_index(drop=True)
# Reassign _row_id after resetting the index
data["_row_id"] = data.index

In [5]:
# Validate feature ranges before removing duplicates
INT_RANGES = {
    "popularity":     (0, 100),
    "duration_ms":    (1, np.inf),
    "key":            (-1, 11),
    "mode":           (0, 1),
    "time_signature": (3, 7),
}
FLOAT_RANGES = {
    "danceability":     (0.0, 1.0),
    "energy":           (0.0, 1.0),
    "speechiness":      (0.0, 1.0),
    "acousticness":     (0.0, 1.0),
    "instrumentalness": (0.0, 1.0),
    "liveness":         (0.0, 1.0),
    "valence":          (0.0, 1.0),
    "loudness":         (-60.0, 0.0),
    "tempo":            (1.0, 250.0),
}

for col, (low, high) in {**INT_RANGES, **FLOAT_RANGES}.items():
    data[col] = pd.to_numeric(data[col], errors="coerce")
    data.loc[~data[col].between(low, high), col] = np.nan

In [6]:
# Impute categorical features using the mode and continuous features using the median
CAT_COLS = ["key", "mode", "time_signature"]
CONT_COLS = list(FLOAT_RANGES) + ["popularity", "duration_ms"]

for col in CAT_COLS:
    data[col] = data[col].fillna(data[col].mode()[0]).astype("Int64")

for col in CONT_COLS:
    data[col] = data[col].fillna(data[col].median())

# Round popularity before casting to Int64, since it represents an integer score
data["popularity"] = data["popularity"].round().astype("Int64")

In [7]:
# Collect all genres associated with each track_id before removing duplicates
genres_by_track = (
    data.groupby("track_id")["track_genre"]
    .agg(lambda s: ";".join(sorted(set(s.dropna()))))
    .rename("all_genres")
)

# Remove duplicate track_ids, using a deterministic tie-breaking strategy
data = (
    data.sort_values(
        ["track_id", "popularity", "_row_id"],
        ascending=[True, False, True],
        kind="mergesort",  # Stable sorting; _row_id ensures deterministic ordering
    )
    .drop_duplicates("track_id", keep="first")
    .reset_index(drop=True)
)

# Merge the complete genre list collected for each track_id
data = data.merge(genres_by_track, left_on="track_id", right_index=True, how="left")

In [8]:
# Remove remaining duplicates using feature fingerprints after removing duplicates with track_id
fingerprint_cols = [
    "duration_ms", "danceability", "energy", "loudness",
    "speechiness", "acousticness", "instrumentalness",
    "liveness", "valence", "tempo",
]

# Round duration_ms to the nearest integer before generating the fingerprint
data["_fp"] = (
    data[fingerprint_cols]
    .assign(duration_ms=lambda d: d["duration_ms"].round(0).astype("int64"))
    .round(3)
    .astype(str)
    .agg("|".join, axis=1)
)

# Combine genres from all track_ids sharing the same feature fingerprint
data["_all_genres_fp"] = data.groupby("_fp")["all_genres"].transform(
    lambda s: ";".join(sorted({g for x in s.dropna() for g in x.split(";") if g}))
)

data = (
    data.sort_values(
        ["_fp", "popularity", "track_id"],
        ascending=[True, False, True],
        kind="mergesort",
    )
    .drop_duplicates("_fp", keep="first")
    .reset_index(drop=True)
)

data["all_genres"] = data["_all_genres_fp"]
data = data.drop(columns=["_fp", "_all_genres_fp"])

In [9]:
# Apply final data types and convert track duration to minutes
data["duration_m"] = (data["duration_ms"] / 60000).round(1)
data = data.drop(columns=["duration_ms"])
data["explicit"] = data["explicit"].astype("int8")
data = data.drop(columns=["_row_id"])

# Arrange columns in the desired output order
meta_order = [
    "track_id", "track_name", "artists", "album_name",
    "track_genre", "all_genres", "popularity", "duration_m", "explicit",
]
audio_order = [
    "danceability", "energy", "key", "loudness", "mode",
    "speechiness", "acousticness", "instrumentalness",
    "liveness", "valence", "tempo", "time_signature",
]
data = data[meta_order + audio_order]

In [10]:
OUT.parent.mkdir(parents=True, exist_ok=True)
# Export the cleaned dataset without writing the DataFrame index
data.to_csv(OUT, index=False)

print(f"Cleaned rows: {len(data):,}")
print(f"Unique track_ids: {data['track_id'].nunique():,}")
print(f"Columns written: {len(data.columns)}")

Cleaned rows: 83,456
Unique track_ids: 83,456
Columns written: 21
